In [ ]:
import sys
print(sys.version)
print(sys.executable)

import json
import numpy as np
import pandas as pd
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger

from ase import Atoms
from dscribe.descriptors import SOAP

RDLogger.DisableLog("rdApp.*")

# =========================
# Config
# =========================
INPUT_FILE = "SMILES_ORIGINAL.csv"
OUTPUT_FILE = "soap_env_vectors.csv"
FAIL_FILE = "soap_env_failures.csv"
META_FILE = "soap_env_metadata.json"

# Polymer handling: "skip" or "cap_to_H"
POLYMER_STAR_MODE = "skip"     # change to "cap_to_H" if you want to keep them

# RDKit 3D settings
N_CONFORMERS = 1              # increase for conformer averaging (slower)
RANDOM_SEED = 0
DO_UFF_OPT = True
MAX_UFF_ITERS = 200

# SOAP settings
R_CUT = 3.0
N_MAX = 6
L_MAX = 4
SIGMA = 0.5


def load_smiles(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.lower().strip() for c in df.columns]

    if "smiles" not in df.columns:
        cols = list(df.columns)
        if len(cols) < 2:
            raise ValueError("Need a 'smiles' column or at least 2 columns (name, smiles).")
        df = df.rename(columns={cols[0]: "name", cols[1]: "smiles"})

    if "name" not in df.columns:
        df["name"] = ""

    df = df.dropna(subset=["smiles"]).copy()
    df["smiles"] = df["smiles"].astype(str).str.strip()
    df["name"] = df["name"].astype(str).str.strip()

    df = df[~df["smiles"].str.lower().isin({"n/a", "na", "-", "?", "/"})].reset_index(drop=True)
    return df


def maybe_handle_polymer_star(smiles: str):
    """Return (smiles_fixed, reason_if_dropped_or_None)."""
    if "*" not in smiles:
        return smiles, None

    if POLYMER_STAR_MODE == "cap_to_H":
        # cheap cap: replace attachment points with H
        return smiles.replace("*", "[H]"), None

    return None, "contains_*"


def smiles_to_3d_mol(smiles: str, n_confs: int = 1, seed: int = 0):
    """
    Return (mol_with_H, conf_ids, reason_or_None).
    """
    smiles_fixed, drop_reason = maybe_handle_polymer_star(smiles)
    if smiles_fixed is None:
        return None, [], drop_reason

    mol = Chem.MolFromSmiles(smiles_fixed)
    if mol is None:
        return None, [], "parse_fail"

    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = int(seed)
    params.useRandomCoords = True

    conf_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=int(n_confs), params=params))
    if not conf_ids:
        return None, [], "embed_failed"

    if DO_UFF_OPT:
        for cid in conf_ids:
            try:
                AllChem.UFFOptimizeMolecule(mol, confId=cid, maxIters=int(MAX_UFF_ITERS))
            except Exception:
                pass

    return mol, conf_ids, None


def rdkit_conf_to_ase_atoms(mol, conf_id: int) -> Atoms:
    conf = mol.GetConformer(conf_id)
    symbols = [a.GetSymbol() for a in mol.GetAtoms()]
    positions = np.array(
        [[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z]
         for i in range(mol.GetNumAtoms())],
        dtype=float
    )
    return Atoms(symbols=symbols, positions=positions)


def is_probably_element_symbol(sym: str) -> bool:
    if sym == "*" or sym == "" or sym is None:
        return False
    return sym[0].isalpha()


# ==============
# Run 
# ==============
df = load_smiles(INPUT_FILE)
print(f"Loaded {len(df)} rows from {INPUT_FILE}")

mols, conf_lists, reasons = [], [], []
all_species = set()

for smi in tqdm(df["smiles"], desc="Embedding 3D (RDKit)"):
    mol, conf_ids, reason = smiles_to_3d_mol(smi, n_confs=N_CONFORMERS, seed=RANDOM_SEED)
    mols.append(mol)
    conf_lists.append(conf_ids)
    reasons.append(reason)

    if mol is not None:
        for a in mol.GetAtoms():
            sym = a.GetSymbol().strip()
            if is_probably_element_symbol(sym):
                all_species.add(sym)

keep_mask = [(m is not None and len(cids) > 0) for m, cids in zip(mols, conf_lists)]
bad = len(keep_mask) - sum(keep_mask)
print(f"Dropping {bad} molecules that failed (or were dropped by policy).")

# Save failures
fail_rows = []
for name, smi, keep, reason in zip(df["name"], df["smiles"], keep_mask, reasons):
    if not keep:
        fail_rows.append((name, smi, reason))
pd.DataFrame(fail_rows, columns=["name", "smiles", "reason"]).to_csv(FAIL_FILE, index=False)

# Filter kept
df_kept = df.loc[keep_mask].reset_index(drop=True)
mols_kept = [m for m, k in zip(mols, keep_mask) if k]
conf_lists_kept = [c for c, k in zip(conf_lists, keep_mask) if k]

species = sorted(all_species)
print("SOAP species basis:", species)

soap = SOAP(
    species=species,
    r_cut=R_CUT,
    n_max=N_MAX,
    l_max=L_MAX,
    sigma=SIGMA,
    periodic=False,
    sparse=False,
)

soap_dim = soap.get_number_of_features()
print("SOAP feature dimension:", soap_dim)

# ===============================
# Per-environment extraction 
# ===============================
rows = []

for (name, smi), mol, conf_ids in tqdm(
    list(zip(df_kept[["name", "smiles"]].itertuples(index=False, name=None), mols_kept, conf_lists_kept)),
    desc="Computing SOAP (per-atom)"
):
    n_atoms = mol.GetNumAtoms()
    symbols = [a.GetSymbol() for a in mol.GetAtoms()]

    # Accumulate per-atom SOAP across conformers, then average
    acc = np.zeros((n_atoms, soap_dim), dtype=np.float64)
    n_used = 0

    for cid in conf_ids:
        atoms = rdkit_conf_to_ase_atoms(mol, cid)

        # DScribe SOAP: per-atom descriptors for all positions (atoms)
        per_atom = soap.create(atoms)  # shape: (n_atoms, soap_dim)
        acc += per_atom
        n_used += 1

    if n_used == 0:
        # Shouldn't happen if conf_ids non-empty, but humans love surprises
        continue

    per_atom_avg = (acc / n_used).astype(np.float32)

    for atom_idx in range(n_atoms):
        rows.append({
            "name": name,
            "smiles": smi,
            "atom_index": atom_idx,
            "atom_symbol": symbols[atom_idx],
            "n_confs_used": n_used,
            "vec": per_atom_avg[atom_idx]
        })

X = np.vstack([r["vec"] for r in rows])
feat_cols = [f"soap_env_{i}" for i in range(X.shape[1])]

out_df = pd.DataFrame({
    "name": [r["name"] for r in rows],
    "smiles": [r["smiles"] for r in rows],
    "atom_index": [r["atom_index"] for r in rows],
    "atom_symbol": [r["atom_symbol"] for r in rows],
    "n_confs_used": [r["n_confs_used"] for r in rows],
})
out_df = pd.concat([out_df, pd.DataFrame(X, columns=feat_cols)], axis=1)
out_df.to_csv(OUTPUT_FILE, index=False)

# Save metadata
meta = {
    "input_file": INPUT_FILE,
    "polymer_star_mode": POLYMER_STAR_MODE,
    "rdkit": {
        "n_conformers": N_CONFORMERS,
        "random_seed": RANDOM_SEED,
        "do_uff_opt": DO_UFF_OPT,
        "max_uff_iters": MAX_UFF_ITERS,
    },
    "soap": {
        "r_cut": R_CUT,
        "n_max": N_MAX,
        "l_max": L_MAX,
        "sigma": SIGMA,
        "species": species,
        "feature_dim": int(soap_dim),
        "output": "per_atom_environment_vectors (no pooling)"
    },
    "n_total_molecules": int(len(df)),
    "n_kept_molecules": int(len(df_kept)),
    "n_failed_molecules": int(len(fail_rows)),
    "n_environment_rows": int(len(rows)),
}
with open(META_FILE, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", OUTPUT_FILE, "shape:", X.shape)
print("Saved failures:", FAIL_FILE, "n=", len(fail_rows))
print("Saved metadata:", META_FILE)



Loaded 106 rows from SMILES_ORIGINAL.csv


Embedding 3D (RDKit): 100%|██████████| 106/106 [00:03<00:00, 27.18it/s]


Dropping 2 molecules that failed (or were dropped by policy).
SOAP species basis: ['Br', 'C', 'Cl', 'F', 'H', 'N', 'Na', 'O', 'P', 'S', 'Si']
SOAP feature dimension: 11055


Computing SOAP (per-atom): 100%|██████████| 104/104 [00:00<00:00, 118.54it/s]


Saved: soap_env_vectors.csv shape: (2273, 11055)
Saved failures: soap_env_failures.csv n= 2
Saved metadata: soap_env_metadata.json
